# BSHI training experiments

Reproduces the implemented seven-plane, 25-value-context BSHI recipes. Recorded optimizer and input-recipe trials use this configuration. The eight newly selected parameters and planned 21-value context are specified in ../configs/selected_features.json but have not been integrated or trained.

Run the setup cell first. Full processing and training require locally provisioned datasets; the quick start uses synthetic tests and saved aggregate results.


In [ ]:
# Load the libraries and project settings needed for this step.
from pathlib import Path
import sys, json, os, time, shutil, subprocess, dataclasses
# Find the project from its configuration file rather than a machine-specific path.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'configs' / 'hemodynamics.json').exists())
assert Path(sys.executable).resolve() == (ROOT / '.venv' / 'Scripts' / 'python.exe').resolve(), 'Select the project .venv kernel'
sys.path.insert(0, str(ROOT / 'notebooks'))
import numpy as np, pandas as pd, torch, psutil
import matplotlib.pyplot as plt
# Run the external command and inspect its completion status.
from IPython.display import display
from hemodynamics import bshi_run as R, bshi_notebook as NB, bshi_train as T, bshi_data as D, cluster as C, bshi_progress as P
# Report the CUDA device and available resources before choosing a training run.
smi = subprocess.run(['nvidia-smi', '--query-gpu=name,driver_version,memory.used,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
print('Root:', ROOT)
print('GPU :', smi)
print(f'torch {torch.__version__} | CUDA build {torch.version.cuda} | available {torch.cuda.is_available()} | free disk {shutil.disk_usage(ROOT).free / 1e9:.0f} GB | free RAM {psutil.virtual_memory().available / 1e9:.0f} GB')
print('compact sweep subset:', (ROOT / R.SWEEP_REL / 'MANIFEST.sha256').exists())
assert torch.cuda.is_available(), 'CUDA is not available: run environment/install_torch_cuda.ps1'

## 2. Configuration (the only cell you edit)

| setting | value | what it does |
|---|---|---|
| `MODE` | `'single_gpu'` (default) | **Desktop only.** No laptop, Spark or Java needed. |
| | `'trial_parallel'` | Several independent short runs (for example one per input recipe). Uses the laptop if it is registered, otherwise runs them one after another on the desktop. |
| | `'ddp_2node'` | ONE run split over desktop + laptop. Needs the laptop and a fast link. **No laptop registered: it falls back to `'single_gpu'` by itself.** |
| `TRAIN_ON` | `'pooled'` (default) | Train on VitalDB + MOVER. Tests: pooled, VitalDB only, MOVER only. |
| | `'vitaldb'` | Train on VitalDB. Tests: VitalDB test split and **all** of MOVER (external validation). |
| | `'mover'` | Train on MOVER. Tests: MOVER test split and the VitalDB test split (reverse direction). |
| `VARIANT` | `'eap'` (default) | Input recipe: `map_only`, `art_only`, `wave`, `wave_map`, `wave_couple`, `eap` (all signals + quality plane), `ea` (no PPG). |
| `EPOCHS` | `30` | Maximum. Stops earlier when validation AUPRC has not improved for `patience` (5) epochs. |
| `OVERRIDES` | `{}` | Anything else in `configs/hemodynamics.json` -> `train`, for example `{'lr': 5e-4, 'batch_size': 256}`. |
| `SMOKE` | `False` | `True`: 3-minute test on a small subset. |
| `CONTINUE` | `''` | Run id (a folder name in `models/`) to continue after a crash. Normally leave empty: the same settings give the same run id, so section 6 continues that run by itself. |

The defaults are the measured best (AdamW beta2 0.95, learning rate 0.0011, batch 128, cosine schedule; evidence in `docs/training_decisions.md`). **Changing any setting starts a new run** (new folder in `models/`), the old one is kept.

In [ ]:
MODE = 'trial_parallel'        # 'single_gpu' | 'trial_parallel' | 'ddp_2node'
TRAIN_ON = 'pooled'        # 'pooled' | 'vitaldb' | 'mover'
VARIANT = 'eap'            # map_only | art_only | wave | wave_map | wave_couple | eap | ea
EPOCHS = 30
OVERRIDES = {}
SMOKE = False              # True: the whole notebook in about 3 minutes on a small subset
CONTINUE = ''              # run id to continue, e.g. 'eap_pooled_adamw_1a2b3c4d'; leave '' normally

cfg = R.load_cfg(ROOT, **{'variant': VARIANT, 'train_on': TRAIN_ON, 'epochs': EPOCHS, **OVERRIDES})
# Use the bounded development settings when a short pipeline check is requested.
if SMOKE:
    cfg = dataclasses.replace(cfg, epochs=2, patience=2, num_workers=0, warmup_epochs=0.5)
RID = NB.run_id(cfg, SMOKE)
# Restore the saved configuration when continuing an existing training run.
if CONTINUE:
    ck = torch.load(ROOT / 'models' / CONTINUE / 'last.pt', map_location='cpu', weights_only=False)
    cfg, RID = T.TrainConfig.from_dict(ck['cfg']), CONTINUE
    del ck
OUT = ROOT / 'models' / RID
EFFECTIVE_MODE, why = NB.resolve_mode(ROOT, MODE)
print('run id        :', RID)
print('mode          :', MODE, '->', EFFECTIVE_MODE, '|', why)
print({k: v for k, v in cfg.to_dict().items() if k in ('variant', 'train_on', 'optimizer', 'beta2', 'lr', 'weight_decay', 'batch_size', 'schedule', 'warmup_epochs', 'epochs', 'patience', 'amp', 'sampler', 'grad_clip', 'loss_weights')})
# Inspect existing checkpoints before deciding whether to continue training.
done = OUT / 'epochs.csv'
if done.exists() and (OUT / 'last.pt').exists():
    n = len(pd.read_csv(done))
    print(f'\nThis run already has {n} finished epoch(s): section 6 CONTINUES it from epoch {n + 1} (to start over, change a setting or delete models/{RID}).')
print('\nEverything of this run is saved here:')
display(NB.paths_table(ROOT, RID, cfg.train_on))

## 3. Data
Patients are split 70 / 15 / 15 into train / validation / test, **no patient is in two splits**; every choice (early stopping, settings) uses train and validation only. The table counts the windows of this experiment.

Training reads the **compact store** (`data/derived/bshi/store_<train_on>_v1`): every 2nd training window (neighbours 30 s apart are near-duplicates), all validation and all test windows, saved as ready tensors. That is about 2,000 windows/s. Reading the same windows straight from the beat store is about **20 times slower** (every shuffled window opens a different surgery's file), so the notebook warns you when no store exists for the chosen `TRAIN_ON`. The BSHI images themselves are never saved: they are built from the beat store and the window index.

In [ ]:
summary, info = R.data_summary(ROOT, cfg)
display(summary)
st = NB.store_status(ROOT, cfg.train_on)
# Use the bounded development settings when a short pipeline check is requested.
if SMOKE:
    print('SMOKE: the small sweep subset is used.')
elif st['exists']:
    print(f"fast data: compact store {Path(st['path']).name}, {st['windows']:,} windows, {st['gb']} GB. About {st['windows'] * 0.55 / 2000 / 60:.1f} min per epoch plus ~0.3 min validation (measured 2,000 windows/s).")
else:
    # Display the recorded values for this step.
    print(f"NO compact store for TRAIN_ON='{cfg.train_on}': training from the beat store would run at only ~100 windows/s (days instead of hours).")
    print('Building one:', NB.store_estimate(ROOT, cfg.train_on, getattr(cfg, 'store_train_stride', 2)))
    print('Set BUILD_STORE = True in the next cell (10-20 minutes) if the disk estimate says ok_to_build.')

In [ ]:
BUILD_STORE = False        # only needed when the cell above says "NO compact store"
if BUILD_STORE:
    from hemodynamics import bshi_store as ST
    summ = ST.build_experiment_store(ROOT, cfg.train_on, stride=getattr(cfg, 'store_train_stride', 2), progress=print)
    print('built:', {k: summ[k] for k in ('windows', 'gb', 'seconds')})

## 4. Quick checks
Does the loader keep up with the GPU (about 2,000 windows/s, measured on shuffled windows like in training), and does a 50-step run from scratch lower a finite loss?

In [ ]:
# Run the configured processing step and retain its returned summary.
trainer, data = R.build_trainer(ROOT, cfg, OUT, smoke=SMOKE, epoch_windows=4096 if SMOKE else None, log=print, progress=False, label=RID)
print(f"data: {data['source']} | {data['n_train']:,} training windows, {data['n_val']:,} validation windows | {data['steps_per_epoch']} steps per epoch")
speed = NB.loader_throughput(data['train_loader'], batches=10 if SMOKE else 25)
print('loader speed (shuffled):', speed, 'windows/s')
if not SMOKE and speed < 1000:
    print('WARN  the loader is much slower than the GPU: build the compact store (section 3), otherwise training takes days.')
san = NB.sanity_run(ROOT, cfg, SMOKE)
print('50-step sanity run:', san)
print('PASS  finite loss' if san['finite'] else 'FAIL  loss is not finite')

## 5. Optional: re-run the optimizer / learning-rate sweep
Needs the Spark cluster (desktop worker is enough) and takes about 40 minutes on the compact sweep subset. Leave `RUN_SWEEP = False` to keep the measured configuration.

In [ ]:
# Run the configured processing step and retain its returned summary.
RUN_SWEEP = False
if RUN_SWEEP:
    from hemodynamics import bshi_sweep as W
    os.environ['BSHI_ROOT'] = str(ROOT)
    curves, sugg = W.lr_range_all(ROOT)
    C.cluster_up(ROOT, workers=(C.SIM_WORKERS[0],))
    sp = C.get_spark('gpu', root=ROOT, workers=1)
    sweep = W.run_sweep(sp, ROOT, sugg, partial_path=str(ROOT / 'results' / 'train' / 'sweep_trials_partial.csv'))
    display(W.rank(sweep)[['name', 'stage', 'val_auprc', 'val_auroc', 'seconds']].head(10))
    sp.stop()
else:
    print('sweep skipped; measured configuration in use:', {k: getattr(cfg, k) for k in ('optimizer', 'beta2', 'lr', 'weight_decay', 'batch_size', 'schedule')})

## 6. Train the selected input recipe
**What you see:** one status line that updates in place (epoch, % of the epoch, windows/s, loss, last validation AUPRC / AUROC, GPU use, estimated time to finish) and a plot (training loss, validation score per epoch, speed) refreshed every minute and after every epoch.

**It is also written to disk** in `models/<run id>/live/`, so you can watch without the notebook (PowerShell in the project folder):
- `$env:PYTHONPATH='notebooks'; .\.venv\Scripts\python.exe -m hemodynamics.bshi_progress watch models\<run id>\live` : one line every 10 s.
- Open `models\<run id>\live\progress.html` in a browser: table + plot, refreshes itself every 15 s.
- From another computer through the VPN: `python -m hemodynamics.bshi_progress serve models\<run id>\live --bind <desktop VPN address> --port 8899`, then open `http://<that address>:8899/progress.html` (read-only; never bind it to `0.0.0.0`).

**Stop and continue:** press the **Interrupt** (stop) button once. Within seconds the status says `paused`. Run this cell again to continue from the last **finished** epoch (the unfinished epoch is repeated: at most about 3 minutes lost). The same works after a crash, a closed notebook or a reboot: reopen, run sections 1-3 with the **same settings**, then this cell. If the status line says `NO UPDATE FOR ...`, the run has stopped.

**Mode:** with `MODE = 'single_gpu'` (or a fallback to it) this cell trains. For `'trial_parallel'` / `'ddp_2node'` with a laptop, use section 8 instead.

In [ ]:
TRAINED = False
# Use the execution mode selected from the available local or remote resources.
if EFFECTIVE_MODE != 'single_gpu':
    print(f"Mode is {EFFECTIVE_MODE}: nothing is trained here, use section 8.")
else:
    trainer.progress = P.Progress(OUT / 'live', label=RID, console_every_s=15)
    print('live files:', (OUT / 'live').relative_to(ROOT), '(progress.html, progress.png, progress.json)')
    try:
        # Fit the model or transform using the supplied fitting partition.
        result = trainer.fit(resume=True)
        TRAINED = True
        trainer.progress.finish({'best_epoch': result['best_epoch'], 'best_val_auprc': result['best_auprc']})
        print(f"finished: best epoch {result['best_epoch'] + 1}, best validation AUPRC {result['best_auprc']:.4f}, stopped early: {result['stopped_early']}, {result['seconds']:.0f} s")
    except KeyboardInterrupt:
        trainer.progress.pause()
        print('paused: run this cell again to continue from the last finished epoch')

## 7. Evaluation (the only place the test split is used)
Validation and test metrics with 95% patient-bootstrap confidence intervals (200 resamples), calibration, AUROC by lead time, the 65-75 mmHg gray zone against the rest, and the G0 (current MAP only) / G4 (all features) baselines from `results/eda` on the same windows. After a restart you can run sections 1-4 and then this cell: it loads the best weights of the run from `models/<run id>/best.pt`.

In [ ]:
ready = TRAINED or (OUT / 'best.pt').exists()
if not ready:
    print('Nothing to evaluate yet: finish section 6 first.')
else:
    if not TRAINED:
        trainer.model.load_state_dict(torch.load(OUT / 'best.pt', map_location='cpu', weights_only=False)['model'])
        print('evaluating the saved best weights of', RID)
    res, preds = R.evaluate_run(ROOT, trainer, data, cfg, SMOKE, reps=100 if SMOKE else 200)
    display(R.metrics_table(res))
    main = 'dev_test' if SMOKE else {'pooled': 'pooled_test', 'vitaldb': 'vitaldb_test', 'mover': 'mover_test'}[cfg.train_on]
    extra = res['_extra'][main]
    fig = R.plot_calibration(preds[main], data['shift'])
    plt.show()
    tte = extra['time_to_event_sec'] if 'time_to_event_sec' in extra else extra.time_to_event_sec
    # Exclude or flag missing values before using these measurements.
    display(R.auroc_by_lead_time(preds[main], tte))
    gz = extra['gray_zone'].reindex(preds[main].window_id).to_numpy().astype(bool)
    display(R.gray_zone_table(preds[main], gz))
    cmp_ = R.compare_with_baselines(ROOT, cfg, res)
    display(cmp_)
    print('PASS  test metrics computed' if np.isfinite(res[main]['auroc']) else 'FAIL  no test metrics')

## 8. Laptop, parallel trials, two-machine training (optional)
Everything above works without this section. Here the notebook looks at what the Spark master knows and applies your `MODE`:

| `MODE` | laptop registered | laptop NOT registered |
|---|---|---|
| `'single_gpu'` | trains on the desktop (section 6) | the same |
| `'trial_parallel'` | one trial per GPU, both machines at once | the trials run **one after another on the desktop** (no Spark, no Java) |
| `'ddp_2node'` | prints the exact commands for both machines (one run split over both GPUs) | **falls back to `'single_gpu'`**, section 6 trains |

Trials are short runs on the compact sweep subset (the laptop has a copy): by default one per input recipe, to compare the recipes. A laptop on another network needs the VPN set up first: `docs/REPRODUCING.md`, part B. Over the internet one run split over both GPUs (`ddp_2node`) is usually slower than the desktop alone: the scaling estimate below tells you.

In [ ]:
# Run the configured processing step and retain its returned summary.
status = NB.cluster_status(ROOT)
print('Spark master :', f"running at {status['master']} with {len(status['workers'])} worker(s)" if status['master_running'] else 'not running (not needed for the desktop alone)')
if status['workers']:
    display(pd.DataFrame(status['workers']))
print('laptop       :', f"registered ({status['remote'][0]['host']})" if status['remote'] else 'not registered')
print('MODE         :', MODE, '->', EFFECTIVE_MODE, '|', why)

# Use the execution mode selected from the available local or remote resources.
if EFFECTIVE_MODE == 'ddp_2node':
    from hemodynamics import scaling_estimate as E
    probes = sorted((ROOT / 'results' / 'probe').glob('probe_*.json')) if (ROOT / 'results' / 'probe').exists() else []
    if len(probes) >= 2:
        print(E.format_report(E.run(str(probes[0]), str(probes[1]))))
    else:
        print('No probe files yet: run cluster_probe on both machines first (docs/REPRODUCING.md, step 8.5).')
    for who, cmd in NB.launch_commands(ROOT, cfg, status['master'] or C.master_host(C.load_config(ROOT))).items():
        print(f'\n{who}:\n  {cmd}')
    print('\nWatch rank 0 with:  python -m hemodynamics.bshi_progress watch runs\\ddp_run\\live')

# Use the execution mode selected from the available local or remote resources.
if EFFECTIVE_MODE in ('trial_parallel', 'trial_sequential'):
    TRIAL_VARIANTS = ['map_only', 'art_only', 'wave', 'wave_map', 'wave_couple', 'eap', 'ea']
    TRIAL_EPOCHS = 6
    cfgs = [{'name': f'{v}_s5052', 'opt': 'adamw_b95', 'lr': cfg.lr, 'weight_decay': cfg.weight_decay, 'batch_size': cfg.batch_size, 'variant': v, 'seed': 5052, 'epochs': TRIAL_EPOCHS,
             'epoch_windows': 29000, 'patience': 2} for v in TRIAL_VARIANTS]
    if EFFECTIVE_MODE == 'trial_parallel':
        os.environ['BSHI_ROOT'] = str(ROOT)
        sp = C.get_spark('gpu', root=ROOT)
        trials = C.run_trials(sp, cfgs, fn='hemodynamics.bshi_sweep:run_trial')
        sp.stop()
    else:
        trials = NB.run_trials_local(ROOT, cfgs)
    tt = pd.DataFrame([{**{k: t[k] for k in ('trial_id', 'ok', 'worker')}, **(t['metrics'] or {})} for t in trials]).drop(columns=['curve'], errors='ignore')
    display(tt)
    (ROOT / 'results' / 'train').mkdir(parents=True, exist_ok=True)
    tt.to_csv(ROOT / 'results' / 'train' / f'trials_{time.strftime("%Y%m%d_%H%M%S")}.csv', index=False)

## 9. Save
`results/train/run_<run id>.json` (configuration and every metric) and a model card next to the checkpoints in `models/<run id>/`.

In [ ]:
# Inspect the selected settings and outputs for this notebook step.
if 'res' in globals() and (TRAINED or (OUT / 'best.pt').exists()):
    p = NB.save_results(ROOT, RID, cfg, res, {'mode': MODE, 'effective_mode': EFFECTIVE_MODE, 'smoke': SMOKE, 'best_epoch': int(trainer.stopper.best_epoch), 'history': trainer.history})
    print('saved', p.relative_to(ROOT), '| checkpoints in', OUT.relative_to(ROOT))
else:
    print('nothing to save yet (run sections 6 and 7 first)')